In [ ]:
from regpy.operators import Operator
from regpy.vecsps import UniformGridFcts

import matplotlib.pyplot as plt


In [ ]:
import numpy as np
from numpy.polynomial.legendre import Legendre, leggauss
from regpy.vecsps import NumPyVectorSpace
from scipy.special import lpmv

import scipy.sparse as sp

def integrated_legendre_basis(p, xi):
    p_extended = np.arange(p+1)
    xx, pp = np.meshgrid(xi, p_extended)
    legendrefunc = lpmv(0, pp, xx)
    # Construction of phip
    b = np.zeros((p+1, p+1))
    b[0,0] = -1
    b[-1,0] = 1
    b[np.arange(1,p),np.arange(1,p)] = 1

    scaling = np.zeros((p+1,1))
    scaling[0,:] = 0.5
    scaling[-1,:] = 0.5
    scaling[1:-1,:] = np.sqrt((2*np.arange(1,p).reshape((p-1,1))+1)/2)
    b = scaling * b 

    Phi_der = b @ legendrefunc

    b = np.zeros((p+1, p+1))
    b[0, 1] = -1
    b[0,0] = 1
    b[-1, 0] = 1
    b[-1,1] = 1
    b[np.arange(1,p),np.arange(p-1)] = -1
    b[np.arange(1,p),np.arange(2,p+1)] = 1

    scaling = np.zeros((p+1,1))
    scaling[0,:] = 0.5
    scaling[-1,:] = 0.5
    scaling[1:-1,:] = 1/np.sqrt(2*(2*np.arange(1,p).reshape((p-1,1))+1))
    b = scaling * b 

    Phi = b @ legendrefunc
    return Phi, Phi_der

class OneDimensionalFEM(NumPyVectorSpace):
    """
    1D Finite Element Method space for polynomial degree p.
    """
    def __init__(self, p=3, n_nodes = 50, a=-1, b=1):
        if b<a:
            raise ValueError("Right endpoint must be greater than left endpoint.")
        if not isinstance(n_nodes, int) or n_nodes < 2:
            raise ValueError("Number of nodes n_nodes must be an integer >= 2.")
        self.nodes = np.linspace(a, b, n_nodes+1, endpoint=True)
        self.hj = np.diff(self.nodes)
        self.n_nodes = n_nodes
        self.start_end = (a, b)
        if not isinstance(p, int) or p < 1: 
            raise ValueError("Polynomial degree p must be a positive integer greater or equal to 1.")
        self.p = p

        self.initialize_FEM_data()
        super().__init__((self.nr_dofs,))

    @property
    def is_finite_dimensional(self):
        return True

    def initialize_FEM_data(self):
        self.quad_pts, self.quad_weights = leggauss(self.p+1)
        self.phi, self.phip = integrated_legendre_basis(self.p,self.quad_pts)

        self.nr_quad_pts = self.n_nodes*len(self.quad_pts)
        self.nr_dofs = self.n_nodes * self.p + 1

        self.hj_long = np.repeat(self.hj, len(self.quad_pts))
        left_ends = np.tile(self.nodes[:-1], (len(self.quad_pts), 1))
        offsets   = (0.5 + 0.5 * self.quad_pts[:, None]) * self.hj
        self.global_quad_pts = (left_ends + offsets).T.flatten()
        self.global_quad_weights = np.tile(self.quad_weights, self.n_nodes)* self.hj_long / 2
        # Build DOF2pts and DOF2pts_der matrices
        DOF2pts = sp.lil_matrix((self.nr_quad_pts,self. nr_dofs))
        DOF2pts_der = sp.lil_matrix((self.nr_quad_pts, self.nr_dofs))

        for j in range(self.n_nodes):
            Ij = np.arange(j * self.p, (j+1) * self.p + 1)  # global DOF indices
            I2j = np.arange(j * len(self.quad_weights), (j + 1) * len(self.quad_pts))  # quad point indices
            DOF2pts[I2j[:, None], Ij] = self.phi.T
            DOF2pts_der[I2j[:, None], Ij] = self.phip.T

        # Convert to CSR for efficient matrix operations
        self.DOF2pts = DOF2pts.tocsr()
        self.DOF2pts_der = DOF2pts_der.tocsr()

        # Assemble mass matrix M
        W_diag = sp.diags(self.global_quad_weights)
        self.M = DOF2pts.T @ W_diag @ DOF2pts
        return None
    
    def get_fem_from_pts(self, x):
        if not isinstance(x, np.ndarray) and x.ndim != 1 and len(x) != len(self.global_quad_pts):
            raise TypeError("Input x must be a numpy array with the size of number of global quadrature points.")
        # Compute projection RHS
        b = self.DOF2pts.T @ (self.global_quad_weights * x)

        # Solve for coefficients
        return sp.linalg.spsolve(self.M, b)
    

class LegendreSpace(NumPyVectorSpace):
    """
    Represents a Legendre series in the 1D FEM space.
    """
    def __init__(self, degree=30, x = None, n_nodes = None, w = None):
        if not isinstance(degree, int) and degree < 1:
            raise TypeError("Degree must be a positive integer.")
        self.degree = degree
        if x is None and n_nodes is None:
            raise ValueError("Either x or n_nodes must be specified")
        elif x is None:
            if not isinstance(n_nodes, int) or n_nodes <= 0:
                raise ValueError("n_node must be a positive float.")
            self.n_nodes = n_nodes
            self.x = np.linspace(-1, 1, n_nodes+1, endpoint=True)
            self.w = np.ones_like(self.x)*(self.x[1]-self.x[0])
        else:
            self.x = x/(x[-1]-x[0])*2
            self.w = np.ones_like(self.x) if w is None else w 
            self.n_nodes = len(x)-1
        self.basis = sp.diags(self.w)@self.legendre_basis(degree, self.x)@(np.sqrt(self.n_nodes/2)*sp.eye(self.degree+1))
        print(self.basis.shape)
        super().__init__((degree+1,))

    def coeff2pts(self, coeffs):
        if not coeffs in self:
            raise TypeError("Coefficients must be a 1D numpy array of length degree + 1.")
        return (self.basis @ (coeffs))
    
    def pts2coeff(self, pts):
        if not isinstance(pts, np.ndarray) or pts.ndim != 1 or len(pts) != self.n_nodes + 1:
            raise TypeError("Points must be a 1D numpy array with the size of number of nodes.")
        return self.basis.T @ pts
    
    def ones(self):
        """
        Returns the constant function 1 in the Legendre series space.
        """
        coeffs = self.zeros()
        coeffs[0] = 1
        return coeffs
    
    def legendre_basis(self,p,x):
        p_extended = np.arange(p+1)
        xx, pp = np.meshgrid(x, p_extended)
        legendrefunc = lpmv(0, pp, xx)
        scale = np.sqrt((2*(p_extended)+1)/2)
        return (legendrefunc * scale.reshape((p+1,1))).T


In [ ]:
from regpy.hilbert import HilbertSpace, L2
from regpy.operators import Operator, PtwMultiplication, MatrixMultiplication
from regpy import util

class ScipyInverseSpLU(Operator):
    def __init__(self, domain, mat):
        if len(mat.shape)!=2 and mat.shape[0] != mat.shape[1]:
            raise ValueError(f"The matrix should be square is not of shape {mat.shape}")
        if len(domain.shape) != 1 and domain.shape[0] != mat.shape[0]:
            raise ValueError(f"The shape of the domain {domain.shape} does not match the shape of the given matrix {mat.shape}")
        if not sp.isspmatrix_csc(mat):
            mat = sp.csc_matrix(mat)
        self.mat_inv = sp.linalg.splu(mat)
        super().__init__(domain, domain, linear = True)

    def _eval(self,x):
        return self.mat_inv.solve(x) 

class L2_Legendre(HilbertSpace):
    """L2 Hilbert space on the Legendre discretized space.

    Parameters
    ----------
    vecsp : LegendreSpace
        The Legendre space on which the L2 space is defined.
    """

    def __init__(self, vecsp):
        if not isinstance(vecsp, LegendreSpace):
            raise ValueError(f"The vecsps is not a LegendreSpace but a {type(vecsp)}")
        super().__init__(vecsp)

    @util.memoized_property
    def gram(self):
        return self.vecsp.identity
        return MatrixMultiplication(matrix = self.mat, domain= self.vecsp,codomain=self.vecsp)
    
    # @util.memoized_property
    # def gram_inv(self):
    #     return ScipyInverseSpLU(self.vecsp,self.mat)


class L2_OneDimensionalFEM(HilbertSpace):
    
    def __init__(self, vecsp):
        if not isinstance(vecsp, OneDimensionalFEM):
            raise ValueError(f"The vecsps is not a OneDimensionalFEM but a {type(vecsp)}")
        self.mat = vecsp.M
        super().__init__(vecsp)

    @util.memoized_property
    def gram(self):
        return MatrixMultiplication(self.mat, domain=self.vecsp,codomain=self.vecsp)
    
    @util.memoized_property
    def gram_inv(self):
        return ScipyInverseSpLU(self.vecsp,self.mat)

L2.register(LegendreSpace,L2_Legendre)
L2.register(OneDimensionalFEM,L2_OneDimensionalFEM)



In [ ]:
fem = OneDimensionalFEM(n_nodes=50,a = -1,b = 1)

leg_dom = LegendreSpace(degree=20, x = fem.global_quad_pts, w = fem.global_quad_weights)
# leg_dom = LegendreSpace(degree=20, n_nodes=50)

# print(fem.global_quad_pts,leg_dom.x)
# x = fem.global_quad_pts
# print(x/(x[-1]-x[0])*2)

plt.figure()
plt.imshow(leg_dom.basis.T@leg_dom.basis)
plt.colorbar()

In [ ]:
f = np.sin(10*leg_dom.x)
f = -0.5*leg_dom.x+0.5
coeff = leg_dom.pts2coeff(f)

plt.plot(leg_dom.x,f,color="green")
plt.plot(leg_dom.x,leg_dom.coeff2pts(coeff))
print(leg_dom.ones())
plt.plot(leg_dom.x,leg_dom.coeff2pts(leg_dom.ones()))

In [ ]:
# leg_dom = LegendreSpace(degree=6,n_nodes= 101)

print(sp.csc_matrix((2/leg_dom.n_nodes)**2 * leg_dom.basis.T@leg_dom.basis).shape, leg_dom.shape)

l2 = L2(leg_dom)

x = leg_dom.ones()
# x = leg_dom.zeros()
# x[2] = 1

print(l2.norm(x))
print(l2.norm(coeff))

print(l2.gram_inv(l2.gram(x)),x)


In [ ]:
legdomain = LegendreSpace(degree=30, n_nodes=1000)

# one = legdomain.zeros()
# one[0] = 1
# one[-1] = 1
# plt.plot(legdomain.x, legdomain.coeff2pts(one), label="Legendre Series")
plt.plot(legdomain.x,legdomain.coeff2pts(legdomain.ones()), label="Constant Function")

z = np.sin(-10*legdomain.x**2)
# z = -5*legdomain.x
z_coeffs = legdomain.pts2coeff(z)
plt.figure()
plt.plot(legdomain.x, z, label="Normal Function")
plt.plot(legdomain.x, legdomain.coeff2pts(z_coeffs), label="Approximation Function")
plt.legend()

In [ ]:
domain = OneDimensionalFEM(p=1, n_nodes=51, a=-1, b=1)
import matplotlib.pyplot as plt
plt.spy(domain.M)
np.allclose(domain.M.todense(), domain.M.todense().T)  # Check if M is symmetric
eigvals = sp.linalg.eigsh(domain.M)[0]
print(eigvals)
ones_quad = np.exp(-domain.global_quad_pts**2)  # Example function to project
ones_quad = np.ones_like(domain.global_quad_pts)  # Example function to project
ones = domain.get_fem_from_pts(ones_quad)
print(ones.shape, domain.M.shape)
print(sum(ones * (domain.M @ ones)))
print(domain.nodes[-1]-domain.nodes[0])

In [ ]:
from regpy.vecsps import DirectSum
class FokkerPlanckOp(Operator):
    def __init__(self, sigma, delta_t, fem, legendre_domain, vector_space, B_diff):
        self.fem = fem
        self.sigma = sigma
        self.delta_t = delta_t
        self.legendre_domain = legendre_domain
        self.vector_space = vector_space
        self.B_diff = B_diff
        super().__init__(domain=self.vector_space, codomain=self.fem)

    @staticmethod
    def create_shared(sigma=0.5, delta_t=0.01, p_fem=3, p_legendre=31, n_nodes=51, a=-1, b=1):
        fem = OneDimensionalFEM(p=p_fem, n_nodes=n_nodes, a=a, b=b)
        legendre_domain = LegendreSpace(degree=p_legendre, x = fem.global_quad_pts, w = fem.global_quad_weights)
        vector_space = DirectSum(fem, legendre_domain)
        B_diff = 0.5 * sigma**2 * (fem.DOF2pts_der.T @ sp.diags(fem.global_quad_weights)) @ fem.DOF2pts_der
        return {
            "sigma" : sigma,
            "delta_t" : delta_t,
            "fem": fem,
            "legendre_domain": legendre_domain,
            "vector_space": vector_space,
            "B_diff": B_diff
        }

    def _eval(self, x, differentiate = False):
        """
        Evaluate one time step the Fokker-Planck operator on the input x.
        """
        u, drift = self.domain.split(x)
        drift_pts = self.fem.global_quad_weights * self.legendre_domain.coeff2pts(drift)
        # system  matrix -0.5*sigma^2*u'' + (drift*u)' 
        B = self.B_diff - self.fem.DOF2pts_der.T @ sp.diags(drift_pts) @ self.fem.DOF2pts 
      
        # time step
        system_matrix = (self.fem.M + self.delta_t * B).tocsc()
        self.system_inv = sp.linalg.splu(system_matrix)
        u_next = self.system_inv.solve(self.fem.M @ u)
        if differentiate:
            self.system_matrix = system_matrix
            self.u_j = self.fem.DOF2pts@u_next
        return u_next

    def _derivative(self, h):
        h_u, h_drift = self.domain.split(h)
        h_drift_pts = self.fem.global_quad_weights * self.legendre_domain.coeff2pts(h_drift)
        rhs = self.fem.M @ h_u + (self.delta_t*sp.eye(self.fem.nr_dofs)) @ (self.fem.DOF2pts_der.T @ (h_drift_pts * self.u_j))
        h_u = self.system_inv.solve(rhs)
        return h_u
    
    def _adjoint(self, y):
        y_u = y
        y_step = self.system_inv.solve(y_u,'T')
        
        y_drift_pts = self.u_j * (self.fem.DOF2pts_der @ (self.delta_t*sp.eye(self.fem.nr_dofs)) @ y_step) 
        y_u = self.fem.M.T @ y_step
        return self.domain.join(y_u, self.legendre_domain.pts2coeff(y_drift_pts))

In [ ]:
from regpy.util.operator_tests import test_operator, test_adjoint, test_linearity, test_derivative
shared_data = FokkerPlanckOp.create_shared(sigma=2, delta_t=0.5, p_fem=2, n_nodes=11, a=-1, b=1)
fokker = FokkerPlanckOp(**shared_data)
# test_operator(fokker)
# test_derivative(fokker)
x = fokker.domain.rand()
_, deriv = fokker.linearize(x)
test_linearity(deriv)
test_linearity(deriv.adjoint)
test_adjoint(deriv)

In [ ]:
d = fokker.fem.DOF2pts 
print(d.shape)
m = d@ d.T
print(type(m))
plt.imshow((d.T@d).todense())
plt.colorbar()

In [ ]:
print("Condition Number:", np.linalg.cond(fokker.system_matrix.toarray()))

In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=4, delta_t=1, p_fem=3, p_legendre= 6,  n_nodes=201, a=-5, b=5)
fokker = FokkerPlanckOp(**shared_data)
drift_pts = -fokker.legendre_domain.x
drift = fokker.legendre_domain.pts2coeff(drift_pts)
# drift_pts = -fokker.fem.global_quad_pts
# drift = fokker.fem.get_fem_from_pts(drift_pts)
plt.figure()
# plt.plot(fokker.legendre_domain.x,fokker.fem.DOF2pts @ drift, label="Drift term approximate")
plt.plot(fokker.legendre_domain.x,fokker.legendre_domain.coeff2pts(drift), label="Drift term approximate")
plt.plot(fokker.legendre_domain.x, drift_pts, '--', label="Drift term at quadrature points")
plt.legend()

In [ ]:
print(drift)

In [ ]:



ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)

u_inf_pts = 1 / np.sqrt(2*np.pi) *np.exp(-(fokker.fem.global_quad_pts)**2/2)
# u_inf_pts = np.sqrt(4/2*np.pi) *np.exp(-4*(fokker.fem.global_quad_pts)**2/2)
u_inf = fokker.fem.get_fem_from_pts(u_inf_pts)
scale = ones.T @ fokker.fem.M @ u_inf
u_inf_pts /= scale
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_inf_pts, label="Limit")


u_0_pts = 1 / np.sqrt(0.2*np.pi) *np.exp(-(fokker.fem.global_quad_pts-2)**2/0.2)
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_0_pts, label="Initial condition")
u_0 = fokker.fem.get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)
u_0 /= ones.T @ fokker.fem.M @ u_0
x_plot = fokker.fem.global_quad_pts
plt.plot(x_plot, fokker.fem.DOF2pts @ u_0, label="FEM projection of initial condition")

u_t = np.copy(u_0)
plt.figure()
plt.plot(x_plot, fokker.fem.DOF2pts @ u_t)
Ut = []
for _ in range(500):
    # print("||u_t|| =", ones.T @ fokker.fem.M @ u_t)
    Ut.append(fokker.fem.DOF2pts @ u_t)
    u_t_next = fokker(fokker.domain.join(u_t, drift))
    plt.plot(x_plot[::fokker.fem.p], (fokker.fem.DOF2pts @ u_t_next)[::fokker.fem.p], color = "blue", label="FEM projection")
    u_t = np.copy(u_t_next)
plt.plot(fokker.fem.global_quad_pts, u_inf_pts, color = "orange", label="Limit")
Ut = np.abs(np.array(Ut))
plt.figure()
from matplotlib.colors import LogNorm
plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 500), origin='lower')
# plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 300), origin='lower',norm=LogNorm(vmin=np.min(Ut), vmax=np.max(Ut)))

In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=0.5, delta_t=0.1, p_fem=8, p_legendre=6, n_nodes=201, a=-5, b=5)
fokker = FokkerPlanckOp(**shared_data)


drift_pts = -50*(fokker.legendre_domain.x**3-0.5*fokker.legendre_domain.x) - 0.05
# drift_pts = np.zeros_like(fokker.fem.global_quad_pts)
# drift_pts = - fokker.legendre_domain.x
# drift_pts = np.sin(np.pi * fokker.legendre_domain.x)
drift = fokker.legendre_domain.pts2coeff(drift_pts)
plt.figure()
plt.plot(fokker.legendre_domain.x,fokker.legendre_domain.coeff2pts(drift), label="Drift term approximate")
plt.plot(fokker.legendre_domain.x, drift_pts, '--', label="Drift term at quadrature points")
plt.legend()


shift = 0
u_0_pts = np.zeros_like(fokker.fem.global_quad_pts)
u_0_pts[len(fokker.fem.global_quad_pts) // 2 +shift] = 1.0  # Initial condition: delta function at the center
# u_0_pts = np.exp(-100 * (fokker.fem.global_quad_pts*10+2)**2)
# u_0_pts = 1 / np.sqrt(0.2*np.pi) *np.exp(-(fokker.fem.global_quad_pts-2)**2/0.2)
plt.figure()
plt.plot(fokker.fem.global_quad_pts, u_0_pts, label="Initial condition")
u_0 = fokker.fem.get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(fokker.fem.global_quad_pts)  # Example function to project
ones = fokker.fem.get_fem_from_pts(ones_quad)
u_0 /= ones.T @ fokker.fem.M @ u_0
x_plot = fokker.fem.global_quad_pts
plt.plot(x_plot, fokker.fem.DOF2pts @ u_0, label="FEM projection of initial condition")


In [ ]:
n = 600

u_t = np.copy(u_0)
plt.figure()
plt.plot(x_plot, fokker.fem.DOF2pts @ u_t)
Ut = []

for _ in range(n):
    # print("||u_t|| =", ones.T @ fokker.fem.M @ u_t)
    Ut.append(fokker.fem.DOF2pts @ u_t)
    u_t_next = fokker(fokker.domain.join(u_t, drift))
    plt.plot(x_plot[::fokker.fem.p], (fokker.fem.DOF2pts @ u_t_next)[::fokker.fem.p], label="FEM projection")
    u_t = np.copy(u_t_next)

Ut = np.abs(np.array(Ut))
plt.figure()
from matplotlib.colors import LogNorm
plt.imshow(Ut, aspect='auto', extent=(-5, 5, 0, n), origin='lower', vmin = 0, vmax= 5)
# plt.imshow(Ut, aspect='auto', extent=(-1, 1, 0, 300), origin='lower',norm=LogNorm(vmin=np.min(Ut), vmax=np.max(Ut)))
plt.figure()
plt.plot(x_plot, fokker.legendre_domain.coeff2pts(drift), label="FEM projection at final time")


In [ ]:
u_final = fokker.fem.DOF2pts@ u_t

u_final[u_final<0] = 0
plt.plot(x_plot,u_final)


n_sample = 100
sample = np.random.poisson(lam = u_final, size=(n_sample,u_final.size)).sum(axis = 0)/n_sample

plt.plot(x_plot,sample)


In [ ]:
from regpy.operators.graph_operator import OperatorGraph

def time_dependent_operator_chain(time_step_ops,output_intermediate_solutions=False,separate_parameter_inputs=False):
    r"""Connects operators to form a chain that can be used for time stepping.

    Parameters:
        time_step_ops (list of Operator): List of operators ordered by time step. 
        output_intermediate_solutions (bool, optional): Controls if intermediate solutions are outputted. Defaults to False.
        separate_parameter_inputs (bool, optional): Controls if each operator gets a seperate additional input. Otherwise all get the same input. Defaults to False.

    Returns:
        OperatorGraph: Operator that has the connections depicted below. 
        False, True (Default)
        In0--->Op0--->Op1--->...--->OpN-1--->OpN--->Out0
                /      /             /        /  
               /      /             /        /
             In1     In2          InN       InN+1
        ################################################
        True, True
        In0--->Op0--->Op1--->...--->OpN-1--->OpN--->OutN
               / \    / \           / \      /  
              / Out0 /  Out1       / OutN-1 /
            In1     In2          InN       InN+1
        ################################################
        False, False
        In0--->Op0--->Op1--->...--->OpN-1--->OpN--->Out0
               /      /             /        /  
              /      /             /        /
        In1__/______/_____________/________/
        ################################################
        True, False
        In0--->Op0--->Op1--->...--->OpN-1--->OpN--->OutN
               / \    / \           / \      /  
              / Out0 /  Out1       / OutN-1 /
        In1__/______/_____________/________/
        """
    edges=[((None,[0]),(time_step_ops[0],0))]#connect start value
    edges+=[((time_step_ops[i],[0]),(time_step_ops[i+1],0)) for i in range(len(time_step_ops)-1)]#passing of solution through time
    if(separate_parameter_inputs):
        edges+=[((None,[j+1]),(time_step_op,1)) for j,time_step_op in enumerate(time_step_ops)]#connecting parameters to operators
    else:
        edges+=[((None,[1]),(time_step_op,1)) for j,time_step_op in enumerate(time_step_ops)]#connecting parameter to operators
    if(output_intermediate_solutions):
        edges+=[((time_step_ops[i],[0]),(None,0)) for i in range(len(time_step_ops)-1)]#get output from each operator except the last
    edges.append(((time_step_ops[-1],[0]),(None,0)))#get output from final operator
    return OperatorGraph(time_step_ops,edges)



In [ ]:
shared_data = FokkerPlanckOp.create_shared(sigma=4, delta_t=0.5, p_fem=3, p_legendre=6, n_nodes=201, a=-5, b=5)
time_steps = [FokkerPlanckOp(**shared_data) for _ in range(500)]

op = time_dependent_operator_chain(time_step_ops=time_steps)

In [ ]:
drift_pts = -op.domain.summands[1].x
drift_pts = -50*(op.domain.summands[1].x**3-0.5*op.domain.summands[1].x)
drift = op.domain.summands[1].pts2coeff(drift_pts)

shift = 5
u_0_pts = np.zeros_like(op.domain.summands[0].global_quad_pts)
u_0_pts[len(op.domain.summands[0].global_quad_pts) // 2 +shift] = 1.0  # Initial condition: delta function at the center
u_0 = op.domain.summands[0].get_fem_from_pts(u_0_pts)
ones_quad = np.ones_like(op.domain.summands[0].global_quad_pts)  # Example function to project
ones = op.domain.summands[0].get_fem_from_pts(ones_quad)
u_0 /= ones.T @ op.domain.summands[0].M @ u_0
print(op.domain.summands)
print(u_0 in op.domain.summands[0], drift in op.domain.summands[1])
u_final_fem = op(op.domain.join(u_0,drift))
u_final = op.domain.summands[0].DOF2pts @ u_final_fem

u_final[u_final<0] = 0
plt.plot(op.domain.summands[0].global_quad_pts,u_final)

In [ ]:
x = op.domain.rand()
_, deriv = op.linearize(x)
test_linearity(deriv)
test_linearity(deriv.adjoint)
test_adjoint(deriv)

In [ ]:
exact_data = u_final
x_plot = op.domain.summands[0].global_quad_pts
plt.plot(x_plot,exact_data, label= "exact data")

n_sample = 100
sample = np.random.poisson(lam = exact_data, size=(n_sample,u_final.size)).sum(axis = 0)/n_sample
plt.plot(x_plot,sample, label = "noisy data")

In [ ]:
from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers import RegularizationSetting
# from regpy.hilbert import L2

setting = RegularizationSetting(
    op = op,
    penalty = L2,
    data_fid = L2
)

solver = IrgnmCG(setting=setting,data = op.codomain.get_fem_from_pts(sample),regpar=0.3)


In [ ]:
one  = op.domain.ones()

print(setting.h_domain.vecsp.summands)
print([s.gram.codomain for s in setting.h_domain.summands])
print(setting.h_domain.norm(one))

In [ ]:
from regpy.stoprules import CountIterations

stop = CountIterations(10)

x,y = solver.run(stop)